 **ScholarSearch - Real Research Paper Search Engine**


by Fahima Abida Chowdhury

ID: 0432220005101135

In [2]:
# ============================================================
#  ScholarSearch — Real Research Paper Search Engine
#  Uses arXiv official API
# ============================================================

# ── 1. Install ───────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "nltk", "gradio", "requests"], check=False)

import re, math, time, requests, xml.etree.ElementTree as ET
from collections import defaultdict
import nltk

for pkg in ("punkt", "stopwords", "punkt_tab"):
    nltk.download(pkg, quiet=True)

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
print("✅ Libraries ready")


✅ Libraries ready


In [3]:
# ── 2. arXiv API fetcher ─────────────────────────────────────
ARXIV_API = "https://export.arxiv.org/api/query"
NS        = "http://www.w3.org/2005/Atom"

CATEGORY_MAP = {
    "cs.AI":"AI", "cs.LG":"Machine Learning", "cs.CL":"NLP",
    "cs.CV":"Computer Vision", "cs.NE":"Neural Networks",
    "cs.RO":"Robotics", "cs.IR":"Information Retrieval",
    "stat.ML":"Statistics/ML", "math.OC":"Optimization",
    "q-bio":"Bioinformatics", "eess":"Signal Processing",
}

def fetch_arxiv(topic: str, max_results: int = 30) -> list[dict]:
    """
    Calls the official arXiv Atom API.
    Always accessible from Google Colab — no scraping, no auth.
    """
    params = {
        "search_query": f"all:{topic}",
        "start":        0,
        "max_results":  max_results,
        "sortBy":       "relevance",
        "sortOrder":    "descending",
    }
    try:
        resp = requests.get(ARXIV_API, params=params, timeout=20)
        resp.raise_for_status()
    except Exception as e:
        return []

    root    = ET.fromstring(resp.text)
    papers  = []
    entries = root.findall(f"{{{NS}}}entry")

    for i, entry in enumerate(entries):
        def tag(name):
            el = entry.find(f"{{{NS}}}{name}")
            return el.text.strip() if el is not None and el.text else ""

        title    = re.sub(r"\s+", " ", tag("title"))
        abstract = re.sub(r"\s+", " ", tag("summary"))
        pub_date = tag("published")[:4]          # "YYYY-MM-DD" → "YYYY"
        year     = int(pub_date) if pub_date.isdigit() else 0

        authors_els = entry.findall(f"{{{NS}}}author")
        authors = ", ".join(
            a.find(f"{{{NS}}}name").text
            for a in authors_els[:4]
            if a.find(f"{{{NS}}}name") is not None
        )
        if len(authors_els) > 4:
            authors += f" +{len(authors_els)-4} more"

        link_el = entry.find(f"{{{NS}}}id")
        link    = link_el.text.strip() if link_el is not None else ""
        # arxiv id → abs URL
        arxiv_id = link.split("/abs/")[-1] if "/abs/" in link else ""
        abs_url  = f"https://arxiv.org/abs/{arxiv_id}" if arxiv_id else link
        pdf_url  = f"https://arxiv.org/pdf/{arxiv_id}" if arxiv_id else ""

        # Primary category
        cat_el   = entry.find("{http://arxiv.org/schemas/atom}primary_category")
        cat_term = cat_el.attrib.get("term", "") if cat_el is not None else ""
        domain   = CATEGORY_MAP.get(cat_term, cat_term)

        papers.append({
            "id":       i + 1,
            "title":    title,
            "abstract": abstract,
            "authors":  authors,
            "year":     year,
            "domain":   domain,
            "link":     abs_url,
            "pdf":      pdf_url,
            "source":   "arXiv",
        })

    return papers

In [4]:
# ── 3. Preprocessor ─────────────────────────────────────────
class Preprocessor:
    def __init__(self):
        self.stemmer    = PorterStemmer()
        self.stop_words = set(stopwords.words("english"))
        self.stop_words.update({
            "also","using","used","use","show","shows","shown",
            "based","new","among","paper","propose","present",
            "method","approach","work","result","results","model",
            "models","data","dataset","task","tasks","learning",
        })

    def process(self, text: str) -> list:
        tokens = re.findall(r"\b[a-z][a-z0-9]*\b", text.lower())
        tokens = [t for t in tokens if t not in self.stop_words and len(t) > 1]
        return [self.stemmer.stem(t) for t in tokens]

In [5]:
# ── 4. Inverted Index ────────────────────────────────────────
class InvertedIndex:
    """
    index[term][doc_id] = {"tf": int, "positions": [int, ...]}
    Built from scratch — no search libraries.
    """
    def __init__(self):
        self.index       = defaultdict(lambda: defaultdict(lambda: {"tf": 0, "positions": []}))
        self.doc_freq    = {}
        self.doc_lengths = {}
        self.N           = 0

    def clear(self):
        self.index.clear()
        self.doc_freq.clear()
        self.doc_lengths.clear()
        self.N = 0

    def add_document(self, doc_id, tokens):
        self.doc_lengths[doc_id] = len(tokens)
        seen = set()
        for pos, term in enumerate(tokens):
            self.index[term][doc_id]["tf"] += 1
            self.index[term][doc_id]["positions"].append(pos)
            seen.add(term)
        for term in seen:
            self.doc_freq[term] = self.doc_freq.get(term, 0) + 1
        self.N += 1

    def tf(self, term, doc_id):
        raw = self.index.get(term, {}).get(doc_id, {}).get("tf", 0)
        return (1 + math.log(raw)) if raw > 0 else 0.0

    def idf(self, term):
        df = self.doc_freq.get(term, 0)
        return math.log((self.N + 1) / (df + 1))   # smoothed IDF

    def tfidf(self, term, doc_id):
        return self.tf(term, doc_id) * self.idf(term)

    def docs_containing(self, term):
        return set(self.index.get(term, {}).keys())

    def stats(self):
        avg = sum(self.doc_lengths.values()) / self.N if self.N else 0
        return {
            "total_documents": self.N,
            "unique_terms":    len(self.index),
            "avg_doc_length":  round(avg, 1),
        }



In [12]:
import re, math, time, requests, xml.etree.ElementTree as ET
from collections import defaultdict
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# ── 5. Search Engine ────────────────
class SearchEngine:
    def __init__(self):
        self.prep   = Preprocessor()
        self.index  = InvertedIndex()
        self.papers = {}

    def build(self, papers: list):
        self.index.clear()
        self.papers.clear()
        for p in papers:
            self.papers[p["id"]] = p
            text   = f"{p['title']} {p['abstract']} {p['authors']} {p['domain']}"
            tokens = self.prep.process(text)
            self.index.add_document(p["id"], tokens)

    def _parse_query(self, raw: str):
        """
        Robust query parsing strategy. Handles explicit operators and processes
        underlying tokens safely to prevent multi-word term extraction errors.
        """
        upper_raw = raw.upper()
        if " AND " in upper_raw:
            mode = "AND"
            # Split by operator, then extract clean constituent words
            segments = re.split(r"\s+AND\s+", raw, flags=re.IGNORECASE)
            terms = []
            for seg in segments:
                terms.extend(self.prep.process(seg))
            return mode, list(dict.fromkeys(terms))  # Deduplicate while preserving order

        if " OR " in upper_raw:
            mode = "OR"
            segments = re.split(r"\s+OR\s+", raw, flags=re.IGNORECASE)
            terms = []
            for seg in segments:
                terms.extend(self.prep.process(seg))
            return mode, list(dict.fromkeys(terms))

        # Default fallback: Space-separated implicit AND
        mode = "AND"
        return mode, self.prep.process(raw)

    def _intersect_scroller(self, postings_lists: list[list[int]]) -> set[int]:
        """
        Implements a classic IR 'Scroller' algorithm (Sorted List Intersection Pointer).
        Walks through sorted list positions sequentially in O(L1 + L2 + ...) time
        without using native high-level set intersection functions.
        """
        if not postings_lists:
            return set()

        # Sort posting lists by size to execute the most restrictive intersections first (Optimization)
        sorted_lists = sorted([sorted(list(p)) for p in postings_lists], key=len)
        result = sorted_lists[0]

        # Intersect sequentially using standard scroller pointers
        for current_list in sorted_lists[1:]:
            intersected = []
            p1, p2 = 0, 0  # Scroller cursors / pointers
            while p1 < len(result) and p2 < len(current_list):
                if result[p1] == current_list[p2]:
                    intersected.append(result[p1])
                    p1 += 1
                    p2 += 1
                elif result[p1] < current_list[p2]:
                    p1 += 1
                else:
                    p2 += 1
            result = intersected
            if not result:
                break
        return set(result)

    def search(self, raw_query, mode_override=None, top_k=10):
        if not raw_query.strip() or not self.papers:
            return []

        mode, processed_terms = self._parse_query(raw_query)
        if mode_override and mode_override != "Auto":
            mode = mode_override

        if not processed_terms:
            return []

        # Boolean retrieval using inverted index posting lookups
        if mode == "AND":
            postings_lists = []
            for t in processed_terms:
                docs = self.index.docs_containing(t)
                if not docs:  # In an AND query, if one term is missing, result size is strictly 0
                    return []
                postings_lists.append(docs)

            # Execute custom scroller-based linear intersection pointer strategy
            candidates = self._intersect_scroller(postings_lists)
        else:
            # Boolean OR: Union strategy
            candidates = set()
            for t in processed_terms:
                candidates |= self.index.docs_containing(t)

        if not candidates:
            return []

        # Custom multi-faceted TF-IDF Ranking Function
        scores = {}
        for doc_id in candidates:
            # Core Textual Metric: Summed TF-IDF scores across query components
            score  = sum(self.index.tfidf(t, doc_id) for t in processed_terms)
            doc    = self.papers[doc_id]

            # Structural Heuristic: Title Match Bonus (Elevates Precision)
            # Evaluated against original raw parts to catch unstemmed key phrases
            raw_query_words = re.split(r"\s+(?:AND|OR)\s+|\s+", raw_query, flags=re.IGNORECASE)
            title_matches = sum(1 for w in raw_query_words if w.lower() and w.lower() in doc["title"].lower())
            score += 1.5 * title_matches

            # Temporal Metric: Normalized Publication Recency Weighting
            score += max(0.0, (doc["year"] - 1990) / 100)

            scores[doc_id] = score

        # Sort candidate matches down by score magnitude
        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        return [
            {**self.papers[doc_id], "score": round(score, 3)}
            for doc_id, score in ranked[:top_k]
        ]

    def stats(self):
        return self.index.stats()

    def top_terms(self, n=25):
        return sorted(
            self.index.doc_freq.items(), key=lambda x: x[1], reverse=True
        )[:n]

engine = SearchEngine()
print("✅ Search engine ready with standard linear scroller intersections")

✅ Search engine ready with standard linear scroller intersections


In [9]:
# ── 6. Helpers ───────────────────────────────────────────────
def make_snippet(text, query_terms, length=240):
    if not text:
        return "<em style='color:var(--color-text-tertiary)'>No abstract available.</em>"
    tl = text.lower()
    best_pos, best_count = 0, 0
    for t in query_terms:
        pos = tl.find(t.lower())
        if pos < 0:
            continue
        window = tl[max(0, pos - 40): pos + 140]
        count  = sum(1 for x in query_terms if x.lower() in window)
        if count > best_count:
            best_count, best_pos = count, max(0, pos - 40)
    prefix = "…" if best_pos > 0 else ""
    suffix = "…" if best_pos + length < len(text) else ""
    snip   = prefix + text[best_pos: best_pos + length] + suffix
    for t in query_terms:
        snip = re.sub(
            f"({re.escape(t)})", r"<mark>\1</mark>", snip, flags=re.IGNORECASE
        )
    return snip


def html_results(results, query):
    if not results:
        return (
            "<div style='text-align:center;padding:40px;"
            "color:var(--color-text-secondary)'>"
            f"<p>No results for <b>{query}</b>.</p>"
            "<p style='margin-top:8px'>Try OR mode, or use fewer / different keywords.</p>"
            "</div>"
        )
    raw_terms = re.split(r"\s+(?:AND|OR)\s+", query, flags=re.IGNORECASE)
    raw_terms = [t.strip() for t in raw_terms if t.strip()]

    html = (
        f"<div style='font-size:13px;color:var(--color-text-secondary);"
        f"margin-bottom:12px;padding-bottom:10px;"
        f"border-bottom:0.5px solid var(--color-border-tertiary)'>"
        f"About <b>{len(results)}</b> result(s) for &ldquo;<b>{query}</b>&rdquo; "
        f"&nbsp;&middot;&nbsp; source: <b>arXiv</b></div>"
    )

    for r in results:
        snip     = make_snippet(r["abstract"], raw_terms)
        year_tag = (
            f"<span style='background:#e6f4ea;color:#1e6e3a;border-radius:10px;"
            f"padding:1px 9px;font-size:11px'>{r['year']}</span>"
            if r["year"] else ""
        )
        dom_tag  = (
            f"<span style='background:#e8f0fe;color:#185fa5;border-radius:10px;"
            f"padding:1px 9px;font-size:11px'>{r['domain']}</span>"
            if r["domain"] else ""
        )
        pdf_link = (
            f"&nbsp;<a href='{r['pdf']}' target='_blank' "
            f"style='font-size:11px;color:#c5221f;text-decoration:none;"
            f"border:0.5px solid #f5c6c6;border-radius:10px;padding:1px 8px'>"
            f"PDF</a>"
            if r.get("pdf") else ""
        )
        score_tag = (
            f"<span style='background:#fce8e6;color:#9a2c2c;border-radius:10px;"
            f"padding:1px 9px;font-size:11px'>Score&nbsp;{r['score']}</span>"
        )
        title_html = (
            f"<a href='{r['link']}' target='_blank' "
            f"style='color:#1a73e8;text-decoration:none'>{r['title']}</a>"
            if r.get("link") else r["title"]
        )
        html += f"""
        <div style='border:0.5px solid var(--color-border-tertiary);border-radius:12px;
                    padding:14px 18px;margin-bottom:14px;
                    background:var(--color-background-primary)'>
          <div style='font-size:17px;color:#1a73e8;font-weight:500;line-height:1.3'>
            {title_html}
          </div>
          <div style='font-size:12px;color:var(--color-text-secondary);
                      margin:5px 0 8px;display:flex;gap:8px;flex-wrap:wrap;align-items:center'>
            <span>{r['authors'] or 'Unknown'}</span>
            {year_tag} {dom_tag} {score_tag} {pdf_link}
          </div>
          <div style='font-size:13px;color:var(--color-text-secondary);line-height:1.65'>
            {snip}
          </div>
        </div>"""
    return html

In [10]:
# ── 7. Gradio callbacks ──────────────────────────────────────
def fetch_and_index(topic, num_fetch):
    if not topic.strip():
        yield "⚠️ Please enter a topic.", ""
        return

    yield f"⏳ Contacting arXiv API for **{topic}**…", ""

    papers = fetch_arxiv(topic.strip(), max_results=int(num_fetch))

    if not papers:
        yield (
            "❌ arXiv returned no results. "
            "Try a different topic or check your connection.", ""
        )
        return

    engine.build(papers)
    s = engine.stats()
    summary = (
        f"✅ Indexed **{s['total_documents']}** arXiv papers about *{topic}*  \n"
        f"📚 **{s['unique_terms']}** unique terms &nbsp;·&nbsp; "
        f"avg doc length **{s['avg_doc_length']}** tokens  \n\n"
        f"➡️ Go to the **Search** tab and start querying!"
    )
    # Show fetched titles
    titles_html = "<br>".join(
        f"<span style='color:#1a73e8'>{p['id']}.</span> {p['title']} "
        f"<span style='color:#70757a;font-size:12px'>({p['year']})</span>"
        for p in papers
    )
    yield summary, titles_html


def do_search(query, mode, top_k):
    if not engine.papers:
        return "<p style='padding:20px;color:var(--color-text-secondary)'>Please fetch papers first (tab ①).</p>"
    if not query.strip():
        return "<p style='padding:20px;color:var(--color-text-secondary)'>Enter a query above.</p>"
    mode_map = {"Auto (AND)": "AND", "AND": "AND", "OR": "OR"}
    results  = engine.search(query, mode_override=mode_map.get(mode, "AND"), top_k=int(top_k))
    return html_results(results, query)


def index_stats():
    if not engine.papers:
        return "<p style='color:var(--color-text-secondary)'>No papers indexed yet. Use tab ① first.</p>"
    s     = engine.stats()
    terms = engine.top_terms(25)
    max_df = terms[0][1] if terms else 1
    rows  = "".join(
        f"<tr>"
        f"<td style='padding:5px 8px'>{i+1}</td>"
        f"<td style='padding:5px 8px'><code>{t}</code></td>"
        f"<td style='padding:5px 8px'>{df}</td>"
        f"<td style='padding:5px 8px'>"
        f"<div style='background:var(--color-background-secondary);"
        f"height:8px;border-radius:4px;width:160px'>"
        f"<div style='background:#1a73e8;height:8px;border-radius:4px;"
        f"width:{int(df/max_df*160)}px'></div></div></td></tr>"
        for i, (t, df) in enumerate(terms)
    )
    return f"""
    <div style='display:grid;grid-template-columns:repeat(3,1fr);gap:12px;margin-bottom:20px'>
      {''.join(
        f"<div style='background:var(--color-background-secondary);border-radius:8px;"
        f"padding:16px;text-align:center'>"
        f"<div style='font-size:26px;font-weight:500;color:#1a73e8'>{v}</div>"
        f"<div style='font-size:12px;color:var(--color-text-secondary)'>{lbl}</div></div>"
        for v, lbl in [
            (s['total_documents'], "Documents"),
            (s['unique_terms'],    "Unique Terms"),
            (s['avg_doc_length'],  "Avg Doc Length"),
        ]
      )}
    </div>
    <p style='font-size:13px;font-weight:500;margin-bottom:8px'>
      Top 25 terms by document frequency</p>
    <table style='width:100%;border-collapse:collapse;font-size:13px'>
      <thead><tr style='background:var(--color-background-secondary)'>
        <th style='padding:6px 8px;text-align:left'>#</th>
        <th style='padding:6px 8px;text-align:left'>Term (stemmed)</th>
        <th style='padding:6px 8px;text-align:left'>DF</th>
        <th style='padding:6px 8px;text-align:left'>Prevalence</th>
      </tr></thead>
      <tbody>{rows}</tbody>
    </table>"""


In [16]:
# ==============================================================================
# ── 8. Gradio UI (Fixed Tab Order with Google-Style Minimalist Search) ────────
# ==============================================================================
import gradio as gr

# Premium modern style minimalistic layout & responsive custom CSS
CSS = """
mark {
    background: #fef08a;
    border-radius: 4px;
    padding: 2px 4px;
    font-weight: 600;
    color: #1e293b;
}
.nav-container {
    background: linear-gradient(135deg, #0f172a 0%, #1e3a8a 100%);
    padding: 30px;
    border-radius: 16px;
    margin-bottom: 28px;
    box-shadow: 0 10px 25px rgba(0,0,0,0.15);
    text-align: center;
}
.nav-title {
    font-size: 3.2rem;
    font-weight: 800;
    letter-spacing: -2px;
    margin: 0;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
}
.nav-subtitle {
    color: #94a3b8;
    font-size: 14px;
    margin-top: 8px;
    font-weight: 400;
}
.tab-instruction {
    background: #f8fafc;
    border-left: 5px solid #3b82f6;
    padding: 16px;
    border-radius: 4px 12px 12px 4px;
    margin-bottom: 24px;
    box-shadow: 0 1px 3px rgba(0,0,0,0.05);
    color: #334155;
    font-size: 14px;
}
.google-search-row {
    max-width: 800px;
    margin: 0 auto 20px auto;
}
"""

with gr.Blocks(css=CSS, title="ScholarSearch — Academic Discovery Engine") as demo:

    # 1. Premium Dynamic Navbar Banner
    gr.HTML(f"""
    <div class='nav-container'>
      <h1 class='nav-title'>
        <span style='color:#4285F4'>S</span><span style='color:#EA4335'>c</span>
        <span style='color:#FBBC05'>h</span><span style='color:#34A853'>o</span>
        <span style='color:#4285F4'>l</span><span style='color:#EA4335'>a</span>
        <span style='color:#FBBC05'>r</span><span style='color:#34A853'>S</span>
        <span style='color:#4285F4'>e</span><span style='color:#EA4335'>a</span>
        <span style='color:#FBBC05'>r</span><span style='color:#34A853'>c</span>
        <span style='color:#4285F4'>h</span>
      </h1>
      <div class='nav-subtitle'>
        🚀 Next-Generation Academic Retrieval Engine &nbsp;·&nbsp; Developed by Fahima &nbsp;·&nbsp; Linear Pointer Scroller Optimized
      </div>
    </div>
    """)

    with gr.Tabs():
        # ── Tab 1: Ingestion Pipeline (MUST BE FIRST) ─────────────────────
        with gr.Tab("📥 ① arXiv Data Ingestion Pipeline"):
            gr.HTML("""
            <div class='tab-instruction'>
                <strong>Step 1: Corpus Generation:</strong> Connects directly to live scientific clusters via the arXiv Atom API network.
                Enter a subject area below to gather raw metadata records and construct the database before searching.
            </div>
            """)
            with gr.Row():
                topic_box = gr.Textbox(
                    placeholder="e.g., large language models / transformer architecture / quantum computing / computer vision",
                    label="Target Academic Research Domain Topic",
                    scale=4,
                )
                num_sl = gr.Slider(
                    5, 100, value=30, step=5,
                    label="Corpus Document Scale Allocation Limit", scale=1
                )
            fetch_btn    = gr.Button("🔍 Ingest Records & Compile Inverted Index Map", variant="primary")
            fetch_status = gr.Markdown("### _Status: Pipeline Standby. Awaiting parameters..._")
            fetched_list = gr.HTML(label="Streaming Ingestion Ledger Output")

            fetch_btn.click(
                fetch_and_index,
                inputs=[topic_box, num_sl],
                outputs=[fetch_status, fetched_list],
            )

        # ── Tab 2: Search Interface (STAYS SECOND WITH GOOGLE LOOK) ───────
        with gr.Tab("🔎 ② Modern Web Search Interface"):
            gr.HTML("""
            <div class='tab-instruction'>
                <strong>Step 2: Information Retrieval:</strong> Queries are evaluated instantly using your customized linear list pointer-scroller.
                Supports explicit logical operators like <code>AND</code> / <code>OR</code> (e.g., <em>"attention AND transformer"</em>).
            </div>
            """)

            # Central Google-Style Search Form Group
            with gr.Column(elem_classes="google-search-row"):
                with gr.Row():
                    q_box = gr.Textbox(
                        placeholder='Search across indexed literature or execute boolean commands...',
                        label="Enter Search Query String",
                        scale=5,
                    )
                    mode_dd = gr.Dropdown(
                        choices=["Auto (AND)", "AND", "OR"],
                        value="Auto (AND)",
                        label="Execution Logic Strategy",
                        scale=1.5,
                    )

                with gr.Row():
                    topk_sl = gr.Slider(
                        1, 50, value=10, step=1,
                        label="Maximum Results Bound (Top-K Display)", scale=3
                    )
                    search_btn = gr.Button("Google Search Clone", variant="primary", scale=1)

            # Render Target Results Section
            results_out = gr.HTML()

            # Operational Triggers
            search_btn.click(do_search, [q_box, mode_dd, topk_sl], results_out)
            q_box.submit(do_search,     [q_box, mode_dd, topk_sl], results_out)

        # ── Tab 3: Analytics Framework ──────────────────────────────────
        with gr.Tab("📊 ③ Diagnostic Metrics & Analytics"):
            gr.HTML("""
            <div class='tab-instruction'>
                <strong>Inverted Index Structural Diagnostics:</strong> Real-time structural system properties showing total indexed terms,
                sparsity matrices, collection sizes ($N$), and global keyword dictionary frequencies.
            </div>
            """)
            inspect_btn = gr.Button("Query Index Matrix Properties", variant="secondary")
            stats_out   = gr.HTML()
            inspect_btn.click(index_stats, outputs=stats_out)

# Universal Direct Launch
demo.launch(share=True)

/tmp/ipykernel_4628/2795806388.py:52: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CSS, title="ScholarSearch — Academic Discovery Engine") as demo:
/usr/local/lib/python3.12/dist-packages/gradio/components/base.py:207: UserWarning: 'scale' value should be an integer. Using 1.5 will cause issues.
  warnings.warn(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://26c993468a160a7168.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
